# What is a diploma worth?

**A guided investigation · Chapter 6 · 30–45 minutes · No Python experience needed**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/intro2causal/blob/main/notebooks_colab/explore-sheepskin_clean.ipynb)

> **The question**  
> Does crossing an exit-exam threshold change diploma receipt and later earnings?

A diploma may signal skills to employers, but graduates also tend to have more schooling and different backgrounds than nongraduates. Clark and Martorell studied students near a Texas high-school exit-exam passing threshold to make a closer comparison. This extract contains score-bin averages rather than individual student records. Each bin records a diploma-receipt rate, average later earnings, and the number of students represented. Passing the exam does not translate perfectly into receiving a diploma, which makes it useful to examine the two links separately: score to credential, and score to earnings. We will mark the passing threshold, inspect both outcomes, and look at how much information supports each point. A large visual difference in a small bin deserves different scrutiny from a stable pattern supported by many students.

### By the end, you will be able to…

- Separate passing an exam, receiving a diploma, and earning more, while checking bin sizes.
- Read and modify three interactive charts using short Python examples.
- Explain one reason a descriptive pattern alone cannot settle this case’s causal question.

### Before you see the evidence

Would a jump in diploma receipt necessarily imply a jump in earnings?

Write two sentences now. You will return to them at the end.

**Your route:** meet the data → inspect three views → modify and try → ask the causal question.

## Your first few minutes in Colab · 3 minutes

You do not need prior Python experience. A **cell** is one block of text or code. Read the text, then press the **▶** beside the next code cell (or Shift+Enter). Its output appears underneath. Run cells in order: later cells use tables created earlier.

Choose **File → Save a copy in Drive** if you want to keep your changes. If a name is “not defined,” run the earlier setup cells again. **Runtime → Restart session and run all** gives you a fresh start. An error does not mean you have damaged the data: this notebook only reads it.

Keep a short learning log in a text cell. Before each chart, make a prediction. Afterward, write one observation with a number and one question that the chart cannot answer.

**Chart controls:** hover for values, click legend labels to hide/show a group, drag to zoom, and double-click to reset. A chart’s menu also offers a reset/autoscale control.

The setup installs Plotly Express if needed. `import` makes a library available; `pd` and `px` are short names for pandas (tables) and Plotly Express (charts).

In [ ]:
%pip install -q "plotly>=6,<7"

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# One consistent chart style; no plotting experience needed.
px.defaults.template = "plotly_white"
px.defaults.color_continuous_scale = "Blues"
px.defaults.color_discrete_sequence = ["#2563eb", "#7c3aed", "#059669", "#64748b"]

## Meet the data · 5 minutes

**One row:** Exit-exam score bin.  
**Stored size:** 46 rows × 14 columns.

| Column | What it records |
|---|---|
| `minscore` | Last-chance exam score relative to the passing threshold (0). |
| `receivehsd` | Proportion receiving a high-school diploma in the score bin, from 0 to 1. |
| `avgearnings` | Score-bin mean earnings in years 7–11, computed on the source unbalanced panel; source dollars. |
| `n` | Stored cell count in childlabor_clean; number of students in the score bin in sheepskin_clean. |

minscore is distance from the passing threshold. Each row is a score bin; n counts students, while person_years supports earnings observations. Do not average bins as though they were individual students.

`read_csv` downloads a table. We call it `df`. `head(5)` shows just five rows, not a separate sample used for the analysis. Square brackets select columns by their names.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/cmg777/intro2causal/main/data/ch6/sheepskin_clean.csv"
df = pd.read_csv(DATA_URL)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")
display(df[['minscore', 'receivehsd', 'avgearnings', 'n']].head(5))

### Pause: what do the rows mean?

Choose one displayed row. Translate it into a sentence using the variable guide. Which values are categories or identifiers? Which are quantities? A numeric storage type does not necessarily make a code a meaningful measurement.

The next table checks types, missing values, and the number of distinct values for the key columns. `isna()` marks missing cells; `sum()` counts them. Missing is not zero. Ask which comparison would lose a row if one of its required values were missing.

In [ ]:
key_columns = ['minscore', 'receivehsd', 'avgearnings', 'n']
checks = pd.DataFrame({"storage_type": df[key_columns].dtypes.astype(str),
                       "missing_rows": df[key_columns].isna().sum(),
                       "distinct_values": df[key_columns].nunique()})
display(checks)
print("The charts below omit missing values required for each plotted quantity.")

### A small summary before plotting

`describe()` summarizes numeric columns: `count` counts available values, `mean` is the average, and `50%` is the median. The median splits ordered values into two halves. Averages of identifiers or arbitrary category codes are not useful descriptions, even if Python calculates them.

For category-like key variables, the second table counts values, including missing entries. Check these labels before interpreting a chart.

In [ ]:
numeric_keys = df[key_columns].select_dtypes(include="number").columns
if len(numeric_keys):
    display(df[numeric_keys].describe().round(2))
for column in key_columns:
    if df[column].nunique() <= 8:
        display(df[column].value_counts(dropna=False).rename("Rows").to_frame())

## Read the chart code · 2 minutes

`px.bar`, `px.scatter`, `px.line`, and `px.box` choose a chart type. `x` and `y` name columns for the axes; `color` distinguishes groups or values. `labels` translates column names for readers. `fig.show()` displays the result.

A table transformation and a plot answer different questions: first decide what each plotted row represents, then choose how to display it. Before running each chart, predict what you expect to see.

In [ ]:
# Display labels: run once; you do not need to memorize this mapping.
labels = {'ability_group': 'Ability group',
 'avgearnings': 'Mean earnings, years 7–11 (source dollars)',
 'census': 'Census year',
 'group': 'State group',
 'minscore': 'Score relative to passing threshold',
 'n': 'Students in score bin',
 'offer': 'Scholarship assignment',
 'person_years': 'Earnings person-years',
 'plan': 'Assigned insurance plan',
 'state_name': 'Simulated state'}

## Visual investigation 1 · First, check the credential link

**Predict:** Does passing coincide with a change in diploma receipt?

receivehsd is a proportion; multiplying by 100 gives a percentage. We mark zero on the centered exam-score axis. Each point is a score-bin average.

In [ ]:
scores = df.copy()
scores["diploma_percent"] = 100 * scores["receivehsd"]
fig = px.scatter(scores, x="minscore", y="diploma_percent", hover_data=["n"], labels={"minscore": "Score relative to passing threshold", "diploma_percent": "Received diploma (%)", "n": "Students in bin"}, title="Passing and receiving a diploma are different")
fig.add_vline(x=0, line_dash="dash")
fig.show()

**Explore the chart:** Hover just to either side of zero. Notice whether every passing student receives a diploma.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

An imperfect link means the threshold changes the probability of a credential. It does not assign the credential perfectly.

</details>

## Visual investigation 2 · Then, inspect later earnings

**Predict:** Is the earnings pattern equally discontinuous?

We change the outcome while keeping the same score axis. These are mean earnings in years 7–11 from the source’s unbalanced panel, not starting salaries.

In [ ]:
fig = px.scatter(df, x="minscore", y="avgearnings", hover_data=["n", "person_years"], labels=labels, title="Earnings around the same threshold")
fig.add_vline(x=0, line_dash="dash")
fig.show()

**Explore the chart:** Compare this pattern with diploma receipt. Do not infer that a noisy or subtle pattern proves there is no effect.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

A credential jump and an earnings jump are separate pieces of evidence. Precision and the assumptions of the cutoff design matter.

</details>

## Visual investigation 3 · Ask how much evidence supports a point

**Predict:** Are score bins equally informative?

A bar chart of n reveals the number of students behind each score bin. The earnings denominator person_years is a different quantity.

In [ ]:
fig = px.bar(df, x="minscore", y="n", labels=labels, title="Unequal numbers of students behind the averages")
fig.add_vline(x=0, line_dash="dash")
fig.show()

**Explore the chart:** Compare sparse bins with the earnings chart. Which averages might be especially noisy?

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Treating every bin equally is not the same as treating every student equally. Weight choice must match the outcome and its denominator.

</details>

## Make it yours · 8 minutes

### 1. Modify an example

Restrict the earnings chart to scores within 10 points of the threshold and state the restriction.

Copy that chart cell before editing, so you can compare the original and modified outputs. Say what changed in your question as well as in your code.

### 2. Try a small investigation

Create a scatter plot of earnings near the cutoff with marker size representing students per bin.

Write a prediction first. The optional starter cell below is safe to leave unchanged; later cells do not depend on your answer.

In [ ]:
# Your investigation goes here.
# Describe your expected pattern before writing code.

<details>
<summary>Hint 1 · choose the right evidence</summary>

Filter minscore using between(-10, 10).

</details>

<details>
<summary>Hint 2 · a Python starting point</summary>

Pass size="n" to px.scatter and keep person_years in hover_data.

</details>

<details>
<summary>Optional worked solution · open after trying</summary>

<pre><code>near = df[df[&quot;minscore&quot;].between(-10, 10)]
fig = px.scatter(near, x=&quot;minscore&quot;, y=&quot;avgearnings&quot;, size=&quot;n&quot;, hover_data=[&quot;person_years&quot;], labels=labels)
fig.add_vline(x=0, line_dash=&quot;dash&quot;)
fig.show()</code></pre>

Marker size communicates available information; it does not itself apply statistical weighting or measure uncertainty.

Copy the code into the starter cell to run it.

</details>

## Optional extension · do weights change the picture?

Weight diploma proportions by the student count n. Do not reuse this weight uncritically for earnings, whose source denominator is person_years. A whole-side average is not a local cutoff estimate.

A **weighted mean** multiplies each value by its weight, adds those products, and divides by the sum of weights. The code below drops rows missing any required value and keeps positive weights. Each group gets its own denominator. This is a descriptive calculation, not an uncertainty estimate or a causal adjustment.

Predict whether weighting will change the comparison. Then compare the two summaries and explain which units receive more influence.

In [ ]:
weighted_data = df.dropna(subset=["receivehsd", "pass_exam", "n"]).copy()
weighted_data = weighted_data[weighted_data["n"] > 0]
weighted_data["product"] = weighted_data["receivehsd"] * weighted_data["n"]
totals = weighted_data.groupby("pass_exam")[["product", "n"]].sum()
weighted_mean = totals["product"] / totals["n"]
unweighted_mean = weighted_data.groupby("pass_exam")["receivehsd"].mean()
comparison = pd.DataFrame({"Equal row weights": unweighted_mean, "Source weights": weighted_mean})
display(comparison.round(3))

## What can you now say? · 4 minutes

Return to your initial prediction. Write a four-sentence evidence note:

1. **I expected…** State your prediction.
2. **I observed…** Use a chart, a comparison, and the correct units.
3. **This does not establish…** Name a causal claim the evidence cannot settle.
4. **To learn more, I would need…** Propose an additional measurement or a more credible comparison.

### Carry this question into the chapter

**Why must a design first establish that crossing the threshold actually changes the probability of receiving a diploma?**

[Continue with Chapter 6: The Wages of Schooling](https://cmg777.github.io/intro2causal/book/_book/notebooks_quarto/06-wages-of-schooling.html#strategy-4-sheepskin-effects-via-rd) · [Return to the case collection](https://cmg777.github.io/intro2causal/data-dictionary.html#ch6-sheepskin-clean-csv)

### Source and preparation notes

Clark and Martorell (2014), The Signaling Value of a High School Diploma (citation only; no verified destination included)

Forty-six score-bin means, not individual records. `n` and `person_years` are denominators/weights for different outcomes; profiles here treat each bin equally.

These exploratory charts are based on the committed CSV. They are not a complete replication of the original study. Keep the source study, this prepared extract, and your descriptive results distinct.